# Ridge Regression (L2 Regularization)

## Problem: Overfitting with Many Features

When you have many features or features are correlated, Linear Regression learns large coefficients that fit noise. Ridge adds a penalty on coefficient magnitude to prevent overfitting.

**Key property:** Ridge shrinks coefficients but never to exactly zero. All features remain.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score

## 1. Load Data & Prepare

Using diabetes dataset: 442 samples, 10 features

In [ ]:
# Load data
X, y = load_diabetes(return_X_y=True)
feature_names = load_diabetes().feature_names

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Data: {X.shape}, Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Baseline: Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Linear Regression - RMSE: {rmse_lr:.2f}, R2: {r2_lr:.4f}")

## 3. Ridge Regression (alpha=1.0)

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)

rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
r2_ridge = r2_score(y_test, y_pred_ridge)

print(f"Ridge (alpha=1.0) - RMSE: {rmse_ridge:.2f}, R2: {r2_ridge:.4f}")
print(f"Improvement: {rmse_lr - rmse_ridge:.2f} RMSE reduction")

## 4. Compare Coefficients

Ridge shrinks ALL coefficients compared to Linear Regression

In [ ]:
comparison = pd.DataFrame({
    'Feature': feature_names,
    'Linear': lr.coef_,
    'Ridge': ridge.coef_,
    'Shrinkage': lr.coef_ - ridge.coef_
})
print(comparison.to_string(index=False))

## 5. Effect of Different Alpha Values

In [ ]:
# Test multiple alphas to find sweet spot
alphas = [0.01, 0.1, 1, 10, 100, 1000]
results = []

for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({'alpha': alpha, 'RMSE': rmse, 'R2': r2})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## 6. Visualize Alpha Effect

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(results_df['alpha'], results_df['RMSE'], 'o-', linewidth=2, markersize=8, color='blue')
ax.set_xscale('log')
ax.set_xlabel('Alpha (regularization strength)')
ax.set_ylabel('RMSE on test set')
ax.set_title('Ridge Performance: RMSE vs Alpha')
ax.grid(True, alpha=0.3)
plt.show()

## 7. Auto-tune with Cross-Validation

In [ ]:
# RidgeCV tests many alphas automatically
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5)
ridge_cv.fit(X_train_scaled, y_train)

y_pred_cv = ridge_cv.predict(X_test_scaled)
rmse_cv = np.sqrt(mean_squared_error(y_test, y_pred_cv))
r2_cv = r2_score(y_test, y_pred_cv)

print(f"Best alpha: {ridge_cv.alpha_:.4f}")
print(f"CV Ridge - RMSE: {rmse_cv:.2f}, R2: {r2_cv:.4f}")

## Summary

**Ridge Regression:**
- Adds L2 penalty: keeps coefficients small
- Never drops features (all coefficients stay non-zero)
- Great for multicollinearity and overfitting
- Use RidgeCV to find best alpha automatically